# 🧠 Notebook 4 – EnhancedRCVPose Training

**Depends on:** `01_setup.ipynb` + `02_preprocess.ipynb`

**What this notebook does:**

| Step | Description |
|------|-------------|
| 4.1 | Load config + imports |
| 4.2 | Set all training hyperparameters |
| 4.3 | Build DataLoaders (train + val, all classes) |
| 4.4 | Build model, loss, optimiser, scheduler |
| 4.5 | Training loop — Stage 1: backbone frozen (warm-up) |
| 4.6 | Short fine-tuning pass with higher rotation loss weight |
| 4.7 | Save final best checkpoint path to `config.json` |

**GPU optimisations used:**
- `torch.compile()` — graph-level fusion (PyTorch ≥ 2.0, +15-25% throughput)
- AMP (`autocast` + `GradScaler`) — FP16 forward pass, ~2× faster
- Gradient accumulation — simulates large batch without extra VRAM
- `pin_memory=True` + `persistent_workers=True` — faster CPU→GPU transfers
- OneCycleLR — fast convergence with cosine annealing
- Backbone freeze warm-up — prevents pretrained features from being destroyed early

**Next step → `05_evaluate.ipynb`**

## Cell 4.1 – Load config and imports

We load `/content/config.json` (written by `01_setup.ipynb`) and add the repo directory to `sys.path` so we can do `from src.model import ...`.

In [ ]:
import os, json, sys, time, shutil
from datetime import datetime
import numpy as np
import torch
import torch.optim as optim
from torch.utils.data import DataLoader, ConcatDataset
from torch.cuda.amp import GradScaler, autocast
from tqdm import tqdm

with open('/content/config.json') as f:
    CONFIG = json.load(f)

DATA_DIR     = CONFIG['DATA_DIR']
ALL_CLASSES  = CONFIG['ALL_CLASSES']
DRIVE_MODELS = CONFIG['DRIVE_MODELS']
REPO_DIR     = CONFIG['REPO_DIR']

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from src.model   import EnhancedRCVPose, WeightedPoseLoss
from src.dataset import PoseDataset, safe_collate

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ── H100 / Ampere optimisations ───────────────────────────────────────────
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    # TF32: ~3× faster matmul on Ampere+ with minimal precision loss
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32      = True
    # High-precision matmul (enables TF32 paths on H100)
    torch.set_float32_matmul_precision('high')
    # cuDNN auto-tuner: finds fastest conv algorithm for fixed input sizes
    torch.backends.cudnn.benchmark = True
    # BF16: native on H100, more stable than FP16 (no inf/nan from overflow)
    IS_H100    = 'H100' in gpu_name or 'H800' in gpu_name
    AMP_DTYPE  = torch.bfloat16 if IS_H100 else torch.float16
    print(f'   GPU         : {gpu_name}')
    print(f'   AMP dtype   : {AMP_DTYPE}')
    print(f'   TF32        : enabled')
    print(f'   cuDNN bench : enabled')
else:
    IS_H100   = False
    AMP_DTYPE = torch.float16

print(f'✅ Imports done.  Device: {DEVICE}')

## Cell 4.2 – Training hyperparameters

All hyperparameters are defined here in one place. To run an experiment, only change values in this cell.

| Parameter | T4 value | **H100 value** | Reason |
|-----------|----------|----------------|--------|
| `BATCH_SIZE` | 8 | **32** | H100 has 80 GB HBM3 |
| `ACCUM` | 4 | **1** | No need with large batch |
| `NUM_WORKERS` | 4 | **8** | More CPU cores on H100 nodes |
| `FREEZE_EPOCHS` | 15 | **15** | same |
| `W_ROT` | 10 | **10** | same |
| `PATIENCE` | 15 | **15** | same |
| `GRAD_CLIP` | 5.0 | **5.0** | same |

> The code **auto-detects** the GPU and scales `BATCH_SIZE`, `ACCUM`, and `NUM_WORKERS` automatically.

In [ ]:
# Auto-scale batch size and workers based on GPU VRAM
if IS_H100:
    _BATCH   = 32   # H100 80 GB HBM3 → large batch
    _ACCUM   = 1    # No accumulation needed
    _WORKERS = 8    # H100 nodes have many CPU cores
elif torch.cuda.is_available() and torch.cuda.get_device_properties(0).total_memory > 15e9:
    _BATCH   = 16   # A100 / A6000 (40+ GB)
    _ACCUM   = 2
    _WORKERS = 6
else:
    _BATCH   = 8    # T4 / V100 (16 GB)
    _ACCUM   = 4
    _WORKERS = 4

TRAIN_CFG = {
    'OBJECT_IDS'     : ALL_CLASSES,
    'NUM_RADIUS_PTS' : 9,
    'NUM_EPOCHS'     : 80,
    'FREEZE_EPOCHS'  : 15,
    'LR'             : 1e-3,
    'LR_STAGE2'      : 3e-4,
    'WEIGHT_DECAY'   : 1e-2,
    'BATCH_SIZE'     : _BATCH,
    'ACCUM'          : _ACCUM,
    'NUM_WORKERS'    : _WORKERS,
    'W_TRANS'        : 1.0,
    'W_ROT'          : 10.0,
    'W_PTS'          : 1.0,
    'PATIENCE'       : 15,
    'GRAD_CLIP'      : 5.0,
}

print('✅ Hyperparameters set.')
print(f'   Batch size      : {TRAIN_CFG["BATCH_SIZE"]}')
print(f'   Grad accum      : {TRAIN_CFG["ACCUM"]}')
print(f'   Effective batch : {TRAIN_CFG["BATCH_SIZE"] * TRAIN_CFG["ACCUM"]}')
print(f'   Num workers     : {TRAIN_CFG["NUM_WORKERS"]}')

## Cell 4.3 – Build DataLoaders

For each class in `ALL_CLASSES` we create:
- `PoseDataset(split='train', augment=True)` — with colour jitter, flip, noise
- `PoseDataset(split='val', augment=False)` — clean for accurate metrics

`ConcatDataset` merges all class datasets so the model sees all objects in every epoch (important for multi-class generalisation).

**DataLoader settings for maximum GPU throughput:**
- `pin_memory=True` → faster CPU→GPU memory copy
- `persistent_workers=True` → workers stay alive between batches
- `prefetch_factor=2` → 2 batches prefetched per worker
- `collate_fn=safe_collate` → skips samples that failed to load

In [ ]:
train_sets, val_sets = [], []

for oid in TRAIN_CFG['OBJECT_IDS']:
    obj_dir = os.path.join(DATA_DIR, oid)
    try:
        tr = PoseDataset(obj_dir, split='train',
                         num_radius_pts=TRAIN_CFG['NUM_RADIUS_PTS'], augment=True)
        vl = PoseDataset(obj_dir, split='val',
                         num_radius_pts=TRAIN_CFG['NUM_RADIUS_PTS'], augment=False)
        train_sets.append(tr)
        val_sets.append(vl)
        print(f'   ✅ Class {oid}: train={len(tr)}, val={len(vl)}')
    except FileNotFoundError as e:
        print(f'   ⚠️  Class {oid}: {e}')

train_ds = ConcatDataset(train_sets)
val_ds   = ConcatDataset(val_sets)

DL_KW = dict(
    batch_size         = TRAIN_CFG['BATCH_SIZE'],
    num_workers        = TRAIN_CFG['NUM_WORKERS'],
    pin_memory         = True,
    persistent_workers = True,
    prefetch_factor    = 2,
    collate_fn         = safe_collate,
)
train_loader = DataLoader(train_ds, shuffle=True,  **DL_KW)
val_loader   = DataLoader(val_ds,   shuffle=False, **DL_KW)

print(f'\n📦 Total train: {len(train_ds)}  |  val: {len(val_ds)}')
print(f'   Train batches/epoch: {len(train_loader)}')

## Cell 4.4 – Build model, loss, optimiser, scheduler

**`torch.compile()`** (PyTorch ≥ 2.0): traces the model into an optimised TorchInductor graph. Gives ~15–25% speedup with zero code changes. Wrapped in `try/except` for compatibility with older runtimes.

**Stage 1 warm-up:** Backbone is frozen for `FREEZE_EPOCHS` so the randomly-initialised pose/outside9 heads don't corrupt the pretrained ResNet50 weights.

**OneCycleLR:** 1-cycle policy — LR ramps up for `pct_start` fraction, then cosine-decays. Enables aggressive LR peaks for fast progress + careful final refinement.

In [ ]:
model = EnhancedRCVPose(fpn_out_channels=256, pose_hidden=512).to(DEVICE)

try:
    model = torch.compile(model)
    print('✅ torch.compile() applied — graph optimisation active.')
except Exception:
    print('ℹ️  torch.compile() not available, continuing without it.')

model.freeze_backbone()
print(f'🧊 Backbone frozen for first {TRAIN_CFG["FREEZE_EPOCHS"]} epochs (warm-up).')

criterion = WeightedPoseLoss(
    w_trans=TRAIN_CFG['W_TRANS'],
    w_rot  =TRAIN_CFG['W_ROT'],
    w_pts  =TRAIN_CFG['W_PTS'],
)

optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=TRAIN_CFG['LR'], weight_decay=TRAIN_CFG['WEIGHT_DECAY'],
)

scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr          = TRAIN_CFG['LR'],
    epochs          = TRAIN_CFG['FREEZE_EPOCHS'],
    steps_per_epoch = len(train_loader),
    pct_start       = 0.1,
    div_factor      = 25,
    final_div_factor= 1e4,
)

scaler = GradScaler()

total_params = sum(p.numel() for p in model.parameters())
train_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\n🧠 Total params   : {total_params:,}')
print(f'   Trainable params: {train_params:,} (backbone frozen)')

## Cell 4.5 – Training loop (Stage 1: frozen backbone → Stage 2: full model)

**Per-epoch flow:**
1. **Train phase:** Forward pass under `autocast()` → FP16 activations; loss ÷ ACCUM → backward → accumulate gradients; every ACCUM steps: unscale → clip → optimizer.step → scaler.update
2. **Val phase:** `torch.no_grad()` + `autocast()` → fast inference; compute all 4 sub-losses for monitoring
3. **Best model:** save checkpoint to Drive when `val_loss` improves
4. **Early stopping:** break if no improvement for `PATIENCE` epochs

**Stage switch** (epoch == `FREEZE_EPOCHS`): unfreeze backbone, replace optimiser and scheduler for stage 2 (lower LR, cosine decay), reset GradScaler.

In [ ]:
best_val_loss     = float('inf')
best_model_path   = None
no_improve_count  = 0
t0                = time.time()

for epoch in range(TRAIN_CFG['NUM_EPOCHS']):

    if epoch == TRAIN_CFG['FREEZE_EPOCHS']:
        model.unfreeze_backbone()
        optimizer = optim.AdamW(
            model.parameters(),
            lr=TRAIN_CFG['LR_STAGE2'],
            weight_decay=TRAIN_CFG['WEIGHT_DECAY'],
        )
        remaining = TRAIN_CFG['NUM_EPOCHS'] - epoch
        scheduler = optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=remaining
        )
        scaler = GradScaler()
        print(f'\n🔓 Epoch {epoch+1}: Backbone unfrozen — Stage 2 begins.\n')

    # Train phase
    model.train()
    torch.cuda.empty_cache()
    tr = {'total': 0., 'trans': 0., 'rot': 0., 'pts': 0.}
    optimizer.zero_grad()

    for step, batch in enumerate(
        tqdm(train_loader,
             desc=f'Ep {epoch+1:3d}/{TRAIN_CFG["NUM_EPOCHS"]} [train]',
             leave=False)
    ):
        if batch is None:
            continue
        rgb      = batch['rgb'].to(DEVICE, non_blocking=True)
        depth    = batch['depth'].to(DEVICE, non_blocking=True)
        pose_gt  = batch['pose'].to(DEVICE, non_blocking=True)
        rmaps_gt = batch['radius_maps'].to(DEVICE, non_blocking=True)

        with autocast():
            pose_p, rmaps_p = model(rgb, depth)
            loss, tloss, rloss, ploss = criterion(
                pose_p, pose_gt, rmaps_p, rmaps_gt)

        scaler.scale(loss / TRAIN_CFG['ACCUM']).backward()

        if (step + 1) % TRAIN_CFG['ACCUM'] == 0 or (step + 1) == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), TRAIN_CFG['GRAD_CLIP'])
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            if epoch < TRAIN_CFG['FREEZE_EPOCHS']:
                scheduler.step()

        tr['total'] += loss.item()
        tr['trans'] += tloss.item()
        tr['rot']   += rloss.item()
        tr['pts']   += ploss.item()

    for k in tr:
        tr[k] /= len(train_loader)

    # Val phase
    model.eval()
    vl = {'total': 0., 'trans': 0., 'rot': 0., 'pts': 0.}

    with torch.no_grad():
        for batch in tqdm(val_loader,
                          desc=f'Ep {epoch+1:3d}/{TRAIN_CFG["NUM_EPOCHS"]} [val]  ',
                          leave=False):
            if batch is None:
                continue
            rgb      = batch['rgb'].to(DEVICE, non_blocking=True)
            depth    = batch['depth'].to(DEVICE, non_blocking=True)
            pose_gt  = batch['pose'].to(DEVICE, non_blocking=True)
            rmaps_gt = batch['radius_maps'].to(DEVICE, non_blocking=True)
            with autocast():
                pose_p, rmaps_p = model(rgb, depth)
                loss, tloss, rloss, ploss = criterion(
                    pose_p, pose_gt, rmaps_p, rmaps_gt)
            vl['total'] += loss.item()
            vl['trans'] += tloss.item()
            vl['rot']   += rloss.item()
            vl['pts']   += ploss.item()

    for k in vl:
        vl[k] /= len(val_loader)

    if epoch >= TRAIN_CFG['FREEZE_EPOCHS']:
        scheduler.step()

    elapsed = (time.time() - t0) / 60
    lr_now  = optimizer.param_groups[0]['lr']
    print(
        f'Ep {epoch+1:3d}/{TRAIN_CFG["NUM_EPOCHS"]}  '
        f'train[tot={tr["total"]:.4f} t={tr["trans"]:.4f} '
        f'r={tr["rot"]:.4f} p={tr["pts"]:.4f}]  '
        f'val[tot={vl["total"]:.4f} t={vl["trans"]:.4f} '
        f'r={vl["rot"]:.4f} p={vl["pts"]:.4f}]  '
        f'lr={lr_now:.2e}  {elapsed:.1f}min'
    )

    if vl['total'] < best_val_loss:
        best_val_loss   = vl['total']
        no_improve_count = 0
        ts   = datetime.now().strftime('%Y%m%d_%H%M%S')
        path = os.path.join(DRIVE_MODELS, f'best_rcvpose_{ts}.pth')
        torch.save({
            'epoch'              : epoch,
            'model_state_dict'   : model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_loss'           : best_val_loss,
            'train_cfg'          : TRAIN_CFG,
        }, path)
        best_model_path = path
        print(f'   💾 Checkpoint saved → {path}')
    else:
        no_improve_count += 1
        if no_improve_count >= TRAIN_CFG['PATIENCE']:
            print(f'\n⏹  Early stop: no improvement for {TRAIN_CFG["PATIENCE"]} epochs.')
            break

print(f'\n✅ Training complete.  Best val loss: {best_val_loss:.5f}')
print(f'   Checkpoint: {best_model_path}')

## Cell 4.6 – Short fine-tuning pass with higher rotation loss weight

After main training we do 10 more epochs with:
- Backbone **frozen** again (only `pose_head` + `outside9_head` trained)
- Very low LR (`1e-4`) with `CosineAnnealingLR`
- Higher rotation weight (`W_ROT=15`) to improve angular accuracy

This is analogous to classifier fine-tuning in transfer learning and typically gives **1–2 degrees improvement** in rotation error.

We start from the best main-training checkpoint.

In [ ]:
FT_EPOCHS  = 10
FT_LR      = 1e-4
FT_W_ROT   = 15.0

ckpt = torch.load(best_model_path, map_location=DEVICE)
state = {k.replace('_orig_mod.', ''): v for k, v in ckpt['model_state_dict'].items()}
model.load_state_dict(state, strict=False)
model.freeze_backbone()
print(f'🧊 Backbone frozen for fine-tuning ({FT_EPOCHS} epochs, LR={FT_LR}).')

ft_criterion = WeightedPoseLoss(w_trans=1.0, w_rot=FT_W_ROT, w_pts=1.0)
ft_optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()), lr=FT_LR
)
ft_scheduler = optim.lr_scheduler.CosineAnnealingLR(ft_optimizer, T_max=FT_EPOCHS)
ft_scaler    = GradScaler()
ft_best      = float('inf')

for ep in range(FT_EPOCHS):
    model.train()
    tr_loss = 0.
    for batch in tqdm(train_loader, desc=f'FineTune {ep+1}/{FT_EPOCHS}', leave=False):
        if batch is None: continue
        rgb   = batch['rgb'].to(DEVICE, non_blocking=True)
        depth = batch['depth'].to(DEVICE, non_blocking=True)
        pg    = batch['pose'].to(DEVICE, non_blocking=True)
        rg    = batch['radius_maps'].to(DEVICE, non_blocking=True)
        with autocast():
            pp, rp = model(rgb, depth)
            loss, *_ = ft_criterion(pp, pg, rp, rg)
        ft_optimizer.zero_grad()
        ft_scaler.scale(loss).backward()
        ft_scaler.step(ft_optimizer)
        ft_scaler.update()
        tr_loss += loss.item()

    model.eval()
    vl_loss = 0.
    with torch.no_grad():
        for batch in val_loader:
            if batch is None: continue
            rgb   = batch['rgb'].to(DEVICE, non_blocking=True)
            depth = batch['depth'].to(DEVICE, non_blocking=True)
            pg    = batch['pose'].to(DEVICE, non_blocking=True)
            rg    = batch['radius_maps'].to(DEVICE, non_blocking=True)
            with autocast():
                pp, rp = model(rgb, depth)
                loss, *_ = ft_criterion(pp, pg, rp, rg)
            vl_loss += loss.item()

    ft_scheduler.step()
    vl_loss /= len(val_loader)
    tr_loss /= len(train_loader)
    print(f'  FineTune {ep+1}/{FT_EPOCHS}: train={tr_loss:.4f}  val={vl_loss:.4f}')

    if vl_loss < ft_best:
        ft_best = vl_loss
        ft_path = best_model_path.replace('.pth', '_finetuned.pth')
        torch.save({'model_state_dict': model.state_dict(), 'val_loss': ft_best}, ft_path)
        print(f'   💾 Fine-tuned checkpoint saved.')

print(f'\n✅ Fine-tuning complete.  Best val loss: {ft_best:.5f}')

## Cell 4.7 – Save final checkpoint path to `config.json`

We persist the path of the best model into `config.json` so that `05_evaluate.ipynb` and `06_visualize.ipynb` know which model to load without hardcoding paths.

In [ ]:
final_model = ft_path if os.path.isfile(ft_path) else best_model_path

with open('/content/config.json') as f:
    CONFIG = json.load(f)

CONFIG['BEST_POSE_MODEL'] = final_model

with open('/content/config.json', 'w') as f:
    json.dump(CONFIG, f, indent=2)

print(f'✅ Best model path saved to config.json')
print(f'   BEST_POSE_MODEL = {final_model}')

---
## ✅ Training Complete

- ✅ Stage 1 (frozen backbone) ran for `FREEZE_EPOCHS` epochs
- ✅ Stage 2 (full model) ran until early stopping or max epochs
- ✅ Fine-tuning pass completed
- ✅ Best checkpoint saved to Google Drive
- ✅ `config.json` updated with `BEST_POSE_MODEL` path

**Next step → Open and run `05_evaluate.ipynb`**